# Pipeline completo en Colab
Ingesta → perfiles → modelo de riesgo. Descarga los datos públicos del INE (~3.5 GB) y la cartografía. Ejecuta las celdas en orden.

In [ ]:
!git clone https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd ParticipacionCiudadanaINE
!pip -q install -r requirements.txt

In [ ]:
%%bash
set -e
mkdir -p data/raw/zips && cd data/raw/zips
for p in 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2009.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2012.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2015.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2018.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2021.zip 2025/08/DatosAbiertos_DECEyEC_ConteosCensales2024.zip; do
  curl -sL -A 'Mozilla/5.0' -O "https://ine.mx/wp-content/uploads/$p"
done
for z in *.zip; do y=$(echo $z | grep -oE '[0-9]{4}'); mkdir -p ../$y && unzip -oq "$z" -d ../$y; done
rm -f *.zip
cd ../../.. && mkdir -p data/geo && cd data/geo
for f in SECCION MUNICIPIO ENTIDAD; do curl -sL -A 'Mozilla/5.0' -O "https://pautas.ine.mx/transparencia/mapas/cob_2025/$f.zip" && mkdir -p $f && unzip -oq $f.zip -d $f && rm $f.zip; done
ls ../raw ../geo

In [ ]:
!python -m src.ingest

In [ ]:
!python -W ignore -m src.profiles --objetivo 2024

In [ ]:
!python -W ignore -m src.risk_model

In [ ]:
!cat data/processed/riesgo_metricas.json | head -c 6000

In [ ]:
!zip -qr processed.zip data/processed
from google.colab import files
files.download('processed.zip')